# Smart Shopping: cameras, processes, and “threading”

## Why two cameras do not need Python `threading` in one script

- **reacTIVision** is its own **process** → uses one camera (e.g. index 0) and sends TUIO `/tuio/2Dobj` over UDP.
- **`hand_tuio_bridge.py`** is another **process** → uses the IRuin / phone webcam (often index 1) and sends `/tuio/2Dcur` (+ swipe objects).

Those two programs run **at the same time** on Windows; they only share the **UDP port** the C# app listens on. That is **multiprocessing by design**, not one Python script with two threads.

## When would you use threads inside Python?

Only if you wanted **one** process to do heavy work in parallel (e.g. capture thread + processing queue). The bridge is already fast enough as a single loop; adding threads adds complexity without fixing “two cameras” (that is two **devices**, two **handles** — still one loop per OpenCV capture).

## IRuin (phone) vs laptop webcam

OpenCV only sees **camera indices** (0, 1, …), not names. If the phone app is **off**, index `1` might be **another** USB camera or nothing useful.

Use the bridge flags:

- `--list-cameras` — see which indices open and frame size.
- `--wait-for-camera` — **wait** until that index shows a **live** non-black image (start IRuin first).
- `--use-dshow` — on Windows, often helps virtual webcams.

Example:

```text
py -3.9 bridge\hand_tuio_bridge.py --list-cameras
py -3.9 bridge\hand_tuio_bridge.py --camera-index 1 --use-dshow --wait-for-camera --tuio-port 3333 --show-preview
```

## Why this notebook was empty

It was only a placeholder; this cell documents the architecture so you do not need a separate “threading” implementation for the two-camera setup.